In [1]:
import pandas as pd

In [2]:
%cd ../analysis/10.hSC3s/

/var2/Works/junhyeong/RXLR_landscape/analysis/10.hSC3s


/home/junhyeong/miniconda3/envs/biopython-env/lib/python3.13/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [3]:
!ls -al

total 8
drwxrwxr-x.  2 junhyeong junhyeong 4096 Feb 26 18:20 .
drwxrwxr-x. 14 junhyeong junhyeong 4096 Feb 26 18:20 ..


In [4]:
hsc = pd.read_csv("../4.WY_tree/RXLR_tree_cluster_by_manual.tsv", sep = "\t")

In [6]:
hsc3 = hsc[hsc["tree_cluster"] == 3]

In [8]:
with open("hsc3.fasta", "w") as f:
    for entry, seq in zip(hsc3["entry"], hsc3["seq"]):
        f.write(f">{entry}\n{seq}\n")

In [12]:
!conda run -n homology-env hmmsearch -T 0 -o hsc3_wy.results ../6.Update_WY_domain/WY_cluster_entries.hmm ./hsc3.fasta

In [11]:
!conda run -n homology-env hmmsearch -h

# hmmsearch :: search profile(s) against a sequence database
# HMMER 3.3.2 (Nov 2020); http://hmmer.org/
# Copyright (C) 2020 Howard Hughes Medical Institute.
# Freely distributed under the BSD open source license.
# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -
Usage: hmmsearch [options] <hmmfile> <seqdb>

Basic options:
  -h : show brief help on version and usage

Options directing output:
  -o <f>           : direct output to file <f>, not stdout
  -A <f>           : save multiple alignment of all hits to file <f>
  --tblout <f>     : save parseable table of per-sequence hits to file <f>
  --domtblout <f>  : save parseable table of per-domain hits to file <f>
  --pfamtblout <f> : save table of hits and domains to file, in Pfam format <f>
  --acc            : prefer accessions over names in output
  --noali          : don't output alignments, so output is smaller
  --notextw        : unlimit ASCII text output line width
  --textw <n>      : set max width of 

In [14]:
from Bio import SearchIO

In [15]:
hsc3_wy = {}

for query in SearchIO.parse("./hsc3_wy.results", "hmmer3-text"):
    for hit in query:
        if hit.bitscore > 0:
            hsc3_wy[hit.id] = len(hit.hsps)

In [19]:
hsc3["wy_count"] = hsc3["entry"].map(hsc3_wy)

/tmp/ipykernel_1945954/1020565880.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  hsc3["wy_count"] = hsc3["entry"].map(hsc3_wy)


In [22]:
hsc3.groupby("wy_count").count()

,entry,spp,WY,LWY,type,seq,seq_cl,str_cl,PFam,tree_cluster
wy_count,,,,,,,,,,
1.0,16,16,16,16,16,16,16,16,16,16
2.0,55,55,55,55,55,55,55,55,55,55
3.0,37,37,37,37,37,37,37,37,37,37
4.0,6,6,6,6,6,6,6,6,6,6
5.0,2,2,2,2,2,2,2,2,2,2
6.0,6,6,6,6,6,6,6,6,6,6
7.0,2,2,2,2,2,2,2,2,2,2
9.0,1,1,1,1,1,1,1,1,1,1


In [23]:
hsc3.groupby("wy_count").count()["entry"].to_csv("hsc3.wy.txt", sep = "\t", index = False )